## Instruction for team mates: 
please dont make any changes or run any cell without my permission.


author: Udhay Reddy

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns


In [4]:
API_TOKEN = "KGAT_f9b58a41fcaaf9d4feb5294e8b8920c4"

In [5]:
!pip install tensorflow

In [6]:
import os
import kagglehub

# 1. Download dataset
path = kagglehub.dataset_download("mariaherrerot/aptos2019")
print("Path to dataset files:", path)

# 2. Inspect the directory tree to see train/val/test folders
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/ ({len(files)} files)")

Path to dataset files: /kaggle/input/datasets/mariaherrerot/aptos2019
aptos2019/ (3 files)
    val_images/ (0 files)
        val_images/ (366 files)
    train_images/ (0 files)
        train_images/ (2930 files)
    test_images/ (0 files)
        test_images/ (366 files)


## The structure of the dataset isn't standard so we need to inspect using this file

In [7]:
import os
print("Contents of path:", os.listdir(path))
train_path = os.path.join(path, "train_images")
print("Contents of train_images:", os.listdir(train_path)[:5])

Contents of path: ['val_images', 'train_images', 'valid.csv', 'test.csv', 'train_1.csv', 'test_images']
Contents of train_images: ['train_images']


In [8]:
import os
import pandas as pd
import tensorflow as tf

IMG_SIZE = (384, 384)
BATCH_SIZE = 16

# 1. Load the CSV files
train_df = pd.read_csv(os.path.join(path, "train_1.csv"))
val_df = pd.read_csv(os.path.join(path, "valid.csv"))

# Inspect columns to be 100% sure of column names (usually 'id_code' and 'diagnosis')
print("Train CSV columns:", train_df.columns.tolist())
print(train_df.head(3))

# 2. Construct full file paths (handling cases where '.png' is omitted in the CSV)
train_img_dir = os.path.join(path, "train_images", "train_images")
val_img_dir = os.path.join(path, "val_images", "val_images")

def get_image_path(img_id, folder):
    img_id = str(img_id)
    if not img_id.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_id += ".png"
    return os.path.join(folder, img_id)

train_paths = [get_image_path(x, train_img_dir) for x in train_df['id_code']]
train_labels = train_df['diagnosis'].values

val_paths = [get_image_path(x, val_img_dir) for x in val_df['id_code']]
val_labels = val_df['diagnosis'].values

# 3. Build the tf.data pipeline
def load_and_preprocess(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    .shuffle(buffer_size=len(train_paths), seed=42)
    .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"Loaded {len(train_paths)} training images and {len(val_paths)} validation images.")

Train CSV columns: ['id_code', 'diagnosis']
        id_code  diagnosis
0  1ae8c165fd53          2
1  1b329a127307          1
2  1b32e1d775ea          4
Loaded 2930 training images and 366 validation images.


In [9]:
!pip install keras-tuner

In [10]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

In [11]:
# import numpy as np
# import keras_tuner as kt
# import tensorflow as tf
# from tensorflow.keras import layers, models
# from tensorflow.keras.applications import DenseNet121
# from tensorflow.keras.applications.densenet import preprocess_input
# from sklearn.utils.class_weight import compute_class_weight

# # 1. Re-create class_weights (needed for tuner.search)
# classes = np.unique(train_labels)
# weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_labels)
# class_weights = dict(zip(classes, weights))

# # 2. Define data_augmentation
# data_augmentation = tf.keras.Sequential([
#     layers.RandomFlip("horizontal_and_vertical"),
#     layers.RandomRotation(0.15),
#     layers.RandomZoom(0.1),
# ])

# # 3. Define the HyperModel
# def build_tunable_densenet(hp):
#     base_model = DenseNet121(
#         weights='imagenet',
#         include_top=False,
#         input_shape=(384, 384, 3)
#     )
#     base_model.trainable = True

#     unfreeze_layer = hp.Choice(
#         'unfreeze_from',
#         values=['conv5_block1_0_bn', 'conv4_block1_0_bn']
#     )

#     freeze = True
#     for layer in base_model.layers:
#         if layer.name == unfreeze_layer:
#             freeze = False
#         layer.trainable = not freeze

#     inputs = layers.Input(shape=(384, 384, 3))
#     x = data_augmentation(inputs)
#     x = preprocess_input(x)
#     x = base_model(x, training=False)

#     x = layers.GlobalAveragePooling2D()(x)
#     x = layers.BatchNormalization()(x)

#     dropout_1 = hp.Float('dropout_1', min_value=0.2, max_value=0.5, step=0.1)
#     x = layers.Dropout(dropout_1)(x)

#     dense_units = hp.Choice('dense_units', values=[128, 256, 512])
#     x = layers.Dense(dense_units, activation='relu')(x)

#     dropout_2 = hp.Float('dropout_2', min_value=0.1, max_value=0.4, step=0.1)
#     x = layers.Dropout(dropout_2)(x)

#     outputs = layers.Dense(5, activation='softmax')(x)
#     model = models.Model(inputs, outputs)

#     lr = hp.Choice('learning_rate', values=[1e-4, 5e-5, 1e-5])
#     wd = hp.Choice('weight_decay', values=[1e-4, 1e-5])

#     model.compile(
#         optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=wd),
#         loss='sparse_categorical_crossentropy',
#         metrics=['accuracy']
#     )
#     return model

In [12]:
import keras_tuner as kt

In [13]:
# # Use Hyperband for fast pruning of underperforming trials
# tuner = kt.Hyperband(
#     build_tunable_densenet,
#     objective='val_loss',
#     max_epochs=10,
#     factor=3,
#     directory='aptos_tuning',
#     project_name='densenet121_finetune'
# )

# stop_early = tf.keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=3,
#     restore_best_weights=True
# )

# # Run the hyperparameter search
# tuner.search(
#     train_ds,
#     validation_data=val_ds,
#     epochs=10,
#     class_weight=class_weights,
#     callbacks=[stop_early]
# )

# # Print best hyperparameters
# best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
# print(f"""
# Optimal Fine-Tuning Parameters Found:
# - Unfreeze from: {best_hps.get('unfreeze_from')}
# - Dense units: {best_hps.get('dense_units')}
# - Dropout 1 / 2: {best_hps.get('dropout_1')} / {best_hps.get('dropout_2')}
# - Learning rate: {best_hps.get('learning_rate')}
# - Weight decay: {best_hps.get('weight_decay')}
# """)
# # Step 3: Train the Final Winning Model
# # Once the tuner finishes identifying the best hyperparameter combination, build the winning model and train it fully for 20–25 epochs:

# # Python
# best_model = tuner.hypermodel.build(best_hps)

# callbacks = [
#     tf.keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1
#     ),
#     tf.keras.callbacks.EarlyStopping(
#         monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
#     ),
#     tf.keras.callbacks.ModelCheckpoint(
#         'densenet121_aptos_tuned_best.keras', monitor='val_loss', save_best_only=True
#     )
# ]

# history = best_model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=25,
#     class_weight=class_weights,
#     callbacks=callbacks
# )

## Testing and tuning is done. The parameters are obatained. Now its time to train the model.

In [14]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds_fast = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_fast = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [15]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# 1. Define data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

# 2. Compute class weights for imbalanced data
classes = np.unique(train_labels)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_labels)
class_weights = dict(zip(classes, weights))

In [16]:
# import keras_tuner as kt
# import tensorflow as tf
# from tensorflow.keras import layers, models
# from tensorflow.keras.applications import DenseNet121
# from tensorflow.keras.applications.densenet import preprocess_input

# def build_tunable_densenet(hp):
#     # 1. Load pre-trained DenseNet-121
#     base_model = DenseNet121(
#         weights='imagenet',
#         include_top=False,
#         input_shape=(384, 384, 3)
#     )
#     base_model.trainable = True

#     # Freeze up to 'conv5_block1_0_bn'
#     freeze = True
#     for layer in base_model.layers:
#         if layer.name == "conv5_block1_0_bn":
#             freeze = False
#         layer.trainable = not freeze

#     # 2. Input, Augmentation & Backbone
#     inputs = layers.Input(shape=(384, 384, 3))
#     x = data_augmentation(inputs)
#     x = preprocess_input(x)
#     x = base_model(x, training=False)

#     x = layers.GlobalAveragePooling2D()(x)
#     x = layers.BatchNormalization()(x)

#     # 3. Tunable Dropout Ratio
#     dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.6, step=0.1)
#     x = layers.Dropout(dropout_rate)(x)

#     # 4. Tunable Number of Dense Layers & Units per Layer
#     num_dense_layers = hp.Int('num_dense_layers', min_value=1, max_value=3, step=1)
#     for i in range(num_dense_layers):
#         units = hp.Choice(f'units_{i}', values=[128, 256, 512, 1024])
#         x = layers.Dense(units, activation='relu')(x)
#         x = layers.Dropout(dropout_rate)(x)

#     outputs = layers.Dense(5, activation='softmax')(x)
#     model = models.Model(inputs, outputs)

#     # 5. Tunable Learning Rate
#     lr = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4, 5e-5, 1e-5])

#     model.compile(
#         optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
#         loss='sparse_categorical_crossentropy',
#         metrics=['accuracy']
#     )
#     return model

# # Initialize Hyperband Tuner
# tuner = kt.Hyperband(
#     build_tunable_densenet,
#     objective='val_loss',
#     max_epochs=12,
#     factor=3,
#     directory='/kaggle/working/aptos_tuning',
#     project_name='densenet121_head_search',
#     overwrite=True
# )

# stop_early = tf.keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=3,
#     restore_best_weights=True
# )

# # Run the search using your cached datasets
# tuner.search(
#     train_ds_fast,
#     validation_data=val_ds_fast,
#     epochs=12,
#     class_weight=class_weights,
#     callbacks=[stop_early]
# )

# # Retrieve and print the best hyperparameters
# best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# print("\n--- Optimal Hyperparameters Found ---")
# print(f"Number of dense layers: {best_hps.get('num_dense_layers')}")
# for i in range(best_hps.get('num_dense_layers')):
#     print(f"  Layer {i+1} units: {best_hps.get(f'units_{i}')}")
# print(f"Dropout ratio: {best_hps.get('dropout_rate'):.2f}")
# print(f"Learning rate: {best_hps.get('learning_rate')}")

In [17]:
# Trial 30 Complete [00h 10m 50s]
# val_loss: 0.647498607635498

# Best val_loss So Far: 0.5473436713218689
# Total elapsed time: 02h 03m 01s

# --- Optimal Hyperparameters Found ---
# Number of dense layers: 1
#   Layer 1 units: 128
# Dropout ratio: 0.30
# Learning rate: 0.001

In [18]:
# final_model = tuner.hypermodel.build(best_hps)

# callbacks = [
#     tf.keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1
#     ),
#     tf.keras.callbacks.EarlyStopping(
#         monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
#     ),
#     tf.keras.callbacks.ModelCheckpoint(
#         '/kaggle/working/densenet121_aptos_tuned_best.keras',
#         monitor='val_loss',
#         save_best_only=True,
#         verbose=1
#     )
# ]

# history = final.fit(
#     train_ds_fast,
#     validation_data=val_ds_fast,
#     epochs=55,
#     class_weight=class_weights,
#     callbacks=callbacks
# )

# # 3. Explicitly save the final model after training finishes
# # (Because restore_best_weights=True is set in EarlyStopping, final_model holds the best weights)
# final_save_path = '/kaggle/working/densenet121_aptos_50ep_final.keras'
# final_model.save(final_save_path)

# print(f"\nModel successfully saved to: {final_save_path}")
# print("Files in /kaggle/working:", os.listdir('/kaggle/working'))

In [25]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

def build_optimal_densenet():
    # 1. Load pre-trained DenseNet-121
    base_model = DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(384, 384, 3)
    )
    base_model.trainable = True

    # Freeze up to 'conv5_block1_0_bn'
    freeze = True
    for layer in base_model.layers:
        if layer.name == "conv5_block1_0_bn":
            freeze = False
        layer.trainable = not freeze

    # 2. Input, Augmentation & Backbone
    inputs = layers.Input(shape=(384, 384, 3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    # 3. Optimal Head Architecture (Dropout: 0.30, 1 Dense Layer with 128 units)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.30)(x)

    outputs = layers.Dense(5, activation='softmax')(x)
    model = models.Model(inputs, outputs)

    # 4. Optimal Learning Rate (0.001)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

final_model = build_optimal_densenet()

# 5. Callbacks for 50-epoch training
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='/kaggle/working/densenet121_aptos_50ep_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# 6. Train for 50 epochs (fixed variable name from 'final' to 'final_model')
history = final_model.fit(
    train_ds_fast,
    validation_data=val_ds_fast,
    epochs=50,
    class_weight=class_weights,
    callbacks=callbacks
)

# 7. Explicitly save the final model
final_save_path = '/kaggle/working/densenet121_aptos_50ep_final.keras'
final_model.save(final_save_path)

print(f"\nModel successfully saved to: {final_save_path}")
print("Files in /kaggle/working:", os.listdir('/kaggle/working'))

Epoch 1/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - accuracy: 0.5463 - loss: 1.5307
Epoch 1: val_loss improved from None to 0.74150, saving model to /kaggle/working/densenet121_aptos_50ep_best.keras

Epoch 1: finished saving model to /kaggle/working/densenet121_aptos_50ep_best.keras
184/184 ━━━━━━━━━━━━━━━━━━━━ 85s 331ms/step - accuracy: 0.6014 - loss: 1.3155 - val_accuracy: 0.7049 - val_loss: 0.7415 - learning_rate: 0.0010
Epoch 2/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - accuracy: 0.6700 - loss: 1.0624
Epoch 2: val_loss improved from 0.74150 to 0.71214, saving model to /kaggle/working/densenet121_aptos_50ep_best.keras

Epoch 2: finished saving model to /kaggle/working/densenet121_aptos_50ep_best.keras
184/184 ━━━━━━━━━━━━━━━━━━━━ 55s 301ms/step - accuracy: 0.6758 - loss: 1.0351 - val_accuracy: 0.7077 - val_loss: 0.7121 - learning_rate: 0.0010
Epoch 3/50
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - accuracy: 0.6647 - loss: 1.0172
Epoch 3: val_loss did not improve from 0.7121

In [26]:
import numpy as np
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

# Get predictions on the validation set
val_preds_probs = final_model.predict(val_ds_fast)
val_preds = np.argmax(val_preds_probs, axis=1)

# Calculate Quadratic Weighted Kappa
qwk = cohen_kappa_score(val_labels, val_preds, weights='quadratic')
print(f"Validation Quadratic Weighted Kappa (QWK): {qwk:.4f}\n")

print("Classification Report:")
print(classification_report(val_labels, val_preds, target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']))

print("Confusion Matrix:")
print(confusion_matrix(val_labels, val_preds))

23/23 ━━━━━━━━━━━━━━━━━━━━ 11s 360ms/step
Validation Quadratic Weighted Kappa (QWK): 0.8517

Classification Report:
               precision    recall  f1-score   support

        No DR       0.99      0.97      0.98       172
         Mild       0.66      0.68      0.67        40
     Moderate       0.71      0.82      0.76       104
       Severe       0.40      0.27      0.32        22
Proliferative       0.55      0.43      0.48        28

     accuracy                           0.81       366
    macro avg       0.66      0.63      0.64       366
 weighted avg       0.80      0.81      0.80       366

Confusion Matrix:
[[166   4   2   0   0]
 [  0  27  11   0   2]
 [  2   6  85   5   6]
 [  0   1  13   6   2]
 [  0   3   9   4  12]]


In [28]:
import os

# Save the final model explicitly (in addition to ModelCheckpoint)
final_model.save('/kaggle/working/densenet121_aptos_deployment.keras')

print("Saved files in /kaggle/working:", os.listdir('/kaggle/working'))

Saved files in /kaggle/working: ['densenet121_aptos_deployment.keras', 'densenet121_aptos_50ep_best.keras', 'densenet121_aptos_50ep_final.keras']
